# 05 — EMIT Spectral Analysis

Analyse the extracted EMIT spectra for Stress, Moderate, and Healthy vegetation.

Outputs:

- mean spectral profiles
- mean ± standard deviation
- absolute spectral magnitude difference
- Euclidean distance matrix
- SAM matrix
- CSV result tables

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import euclidean

DATA_FILE = Path("../outputs/spectral_profiles/EMIT_extracted_spectra.csv")
OUT_FIG = Path("../outputs/figures")
OUT_TAB = Path("../outputs/tables")
OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_TAB.mkdir(parents=True, exist_ok=True)

CLASSES = ["Stress", "Moderate", "Healthy"]

In [ ]:
df = pd.read_csv(DATA_FILE)

metadata_cols = ["Point_Index", "Class", "X", "Y"]
wavelength_cols = [c for c in df.columns if c not in metadata_cols]
wavelengths = np.array([float(c) for c in wavelength_cols])

spectra = {}
mean_s = {}
std_s = {}

for cls in CLASSES:
    arr = df.loc[df["Class"] == cls, wavelength_cols].to_numpy(dtype=float)

    if arr.size == 0:
        raise ValueError(f"No spectra found for class: {cls}")

    spectra[cls] = arr
    mean_s[cls] = np.nanmean(arr, axis=0)
    std_s[cls] = np.nanstd(arr, axis=0)

    print(f"{cls}: {arr.shape[0]} spectra")

In [ ]:
# Mean ± SD spectral profiles
fig, ax = plt.subplots(figsize=(11, 6))

for cls in CLASSES:
    ax.plot(wavelengths, mean_s[cls], linewidth=2, label=cls)
    ax.fill_between(
        wavelengths,
        mean_s[cls] - std_s[cls],
        mean_s[cls] + std_s[cls],
        alpha=0.15,
    )

ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Reflectance")
ax.set_title("Mean EMIT Hyperspectral Profiles")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
fig.savefig(OUT_FIG / "mean_spectral_profiles.png", dpi=300)
plt.show()

In [ ]:
# Absolute magnitude difference
pairs = [
    ("Stress", "Moderate"),
    ("Moderate", "Healthy"),
    ("Stress", "Healthy"),
]

fig, ax = plt.subplots(figsize=(12, 6))

for c1, c2 in pairs:
    diff = np.abs(mean_s[c1] - mean_s[c2])
    ax.plot(wavelengths, diff, linewidth=2, label=f"{c1} vs {c2}")

ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Absolute Spectral Difference")
ax.set_title("Spectral Magnitude Difference Between Vegetation Classes")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
fig.savefig(OUT_FIG / "spectral_magnitude_difference.png", dpi=300)
plt.show()

In [ ]:
# Euclidean distance
n = len(CLASSES)
euclidean_matrix = np.zeros((n, n), dtype=float)

for i, c1 in enumerate(CLASSES):
    for j, c2 in enumerate(CLASSES):
        euclidean_matrix[i, j] = euclidean(mean_s[c1], mean_s[c2])

euclidean_df = pd.DataFrame(
    euclidean_matrix, index=CLASSES, columns=CLASSES
)
euclidean_df.to_csv(OUT_TAB / "euclidean_distance_matrix.csv")
print(euclidean_df)

In [ ]:
# Spectral Angle Mapper (SAM)
def sam_angle(a, b):
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return np.nan
    cosine = np.dot(a, b) / denominator
    cosine = np.clip(cosine, -1, 1)
    return np.degrees(np.arccos(cosine))

sam_matrix = np.zeros((n, n), dtype=float)

for i, c1 in enumerate(CLASSES):
    for j, c2 in enumerate(CLASSES):
        sam_matrix[i, j] = sam_angle(mean_s[c1], mean_s[c2])

sam_df = pd.DataFrame(sam_matrix, index=CLASSES, columns=CLASSES)
sam_df.to_csv(OUT_TAB / "sam_matrix.csv")
print(sam_df)

In [ ]:
# Save class mean and SD spectra
mean_table = pd.DataFrame({"Wavelength_nm": wavelengths})
for cls in CLASSES:
    mean_table[f"{cls}_Mean"] = mean_s[cls]
    mean_table[f"{cls}_SD"] = std_s[cls]

mean_table.to_csv(
    OUT_TAB / "mean_spectral_profiles.csv", index=False
)

print("Analysis outputs saved in:", OUT_FIG.resolve(), OUT_TAB.resolve())